# Cell 1: Setup and Imports

## Description
 Installs and imports transformers, pytorch, and other necessary libraries. Sets up the device for GPU acceleration if available..

In [ ]:
# --- Step 1: Setup Environment & Install Dependencies ---
import os, sys

REPO_URL = "https://github.com/Turkcoder123/tradingbot-ml.git"
CLONE_DIR = "/content/tradingbot-ml"

# Her platformda calisacak basit mantik
# 1) /kaggle/working varsa -> Kaggle
# 2) /content varsa -> Colab, clone dene
# 3) hicbiri yoksa -> Local

if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')
elif os.path.exists('/content'):
    if not os.path.exists(CLONE_DIR):
        os.system(f'git clone --depth 1 {REPO_URL} {CLONE_DIR} > /dev/null 2>&1')
    if os.path.exists(CLONE_DIR):
        os.chdir(CLONE_DIR)

print("Working directory:", os.getcwd())

get_ipython().system('pip install -q transformers accelerate torch scikit-learn pandas numpy matplotlib tqdm joblib')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from tqdm.notebook import trange, tqdm

import torch
import torch.nn as nn
from transformers import TimeSeriesTransformerForPrediction, TimeSeriesTransformerConfig

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import matplotlib.dates as mdates

# --- Reproducibility ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print("Libraries imported and seeds set.")

# --- Device Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Cell 2: Configure Model and Training Hyperparameters

All key parameters are defined here for configuration



In [ ]:
# --- Step 2: Hyperparameter Configuration ---

# --- Data and Model Architecture ---
CONTEXT_LENGTH = 288        # Covers 50-period SMA + multi-timeframe (1 day of 5m bars)
PREDICTION_LENGTH = 6       # Dynamic: 6 steps ahead (30 min), can be adjusted by confidence

# --- Lags Sequence Configuration ---
LAGS_SEQUENCE = [1, 2, 3, 4, 5, 6, 7]  # This is a strong, standard default.

# --- Model Size & Regularization ---
D_MODEL = 64                # Increased to handle 37 input features
ENCODER_LAYERS = 2
DECODER_LAYERS = 2
ENCODER_ATTENTION_HEADS = 4
DECODER_ATTENTION_HEADS = 4
ENCODER_FFN_DIM = 128
DECODER_FFN_DIM = 128
DROPOUT = 0.2               # Increased for better regularization

# --- Training Schedule & Objective ---
EPOCHS = 50
LEARNING_RATE = 1e-4
BATCH_SIZE = 64
PATIENCE = 10
DISTRIBUTION_OUTPUT = "student_t" 

# Cell 3: Data Loading and Preprocessing

## Description:
Loads the EURUSD CSV file and correctly parses the Date and Time columns into a single Timestamp index for proper time-series handling.

In [ ]:
# --- Step 3: Load and Preprocess Data (OHLCV + Multi-Timeframe) ---
import os

# Cell 01 zaten GitHub'dan klonlanmis dizine cd yapti -> Data/EURUSD_5m_10Yea.csv hazir
file_path = 'Data/EURUSD_5m_10Yea.csv'
assert os.path.exists(file_path), f"Veri dosyasi bulunamadi: {os.path.abspath(file_path)}"
print(f'Veri: {os.path.abspath(file_path)}')

df_raw = pd.read_csv(file_path)

# Parse timestamp
df_raw['Timestamp'] = pd.to_datetime(
    df_raw['Date'].astype(str) + ' ' + df_raw['Time'].astype(str),
    format='%Y%m%d %H:%M:%S'
)

# Keep all OHLCV columns
df_5m = df_raw[['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
df_5m.set_index('Timestamp', inplace=True)

# --- Create Higher Timeframes from 5m data ---
# 1 Hour (12 x 5min bars)
df_1h = df_5m.resample('1H').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}).dropna()

# 1 Day (288 x 5min bars)
df_1d = df_5m.resample('1D').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}).dropna()

print(f"5m data: {len(df_5m)} rows")
print(f"1h data: {len(df_1h)} rows")
print(f"1d data: {len(df_1d)} rows")

# Merge higher timeframe data into 5m for feature enrichment
df = df_5m.copy()
df['Close_1h'] = df['Close'].resample('1H').last().reindex(df.index, method='ffill')
df['Close_1d'] = df['Close'].resample('1D').last().reindex(df.index, method='ffill')
df['Volume_1h'] = df['Volume'].resample('1H').sum().reindex(df.index, method='ffill')
df['Volume_1d'] = df['Volume'].resample('1D').sum().reindex(df.index, method='ffill')
df['High_1h'] = df['High'].resample('1H').max().reindex(df.index, method='ffill')
df['Low_1h'] = df['Low'].resample('1H').min().reindex(df.index, method='ffill')
df['High_1d'] = df['High'].resample('1D').max().reindex(df.index, method='ffill')
df['Low_1d'] = df['Low'].resample('1D').min().reindex(df.index, method='ffill')

# Price range features (intrinsic volatility)
df['Range_5m'] = df['High'] - df['Low']
df['Range_1h'] = df['High_1h'] - df['Low_1h']
df['Range_1d'] = df['High_1d'] - df['Low_1d']

# Chronological split
train_size = int(len(df) * 0.60)
val_size = int(len(df) * 0.20)

train_df = df.iloc[:train_size].copy()
val_df = df.iloc[train_size:train_size + val_size].copy()
test_df = df.iloc[train_size + val_size:].copy()

print(f"\nTraining set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

# Cell 4: Feature Engineering and Data Splitting

## Description:
Creates time-based features required by the Transformer for positional information. The data is then split chronologically into train, validation, and test sets (60/20/20)


In [ ]:
# --- Step 4: Feature Engineering with Technical Indicators & Multi-Timeframe Features ---

def create_technical_features(df):
    """Creates comprehensive technical indicators and time features."""
    df_feat = df.copy()
    
    # --- Time-based features (scaled to [-0.5, 0.5]) ---
    df_feat['hour'] = df_feat.index.hour / 23.0 - 0.5
    df_feat['day_of_week'] = df_feat.index.dayofweek / 6.0 - 0.5
    df_feat['day_of_month'] = (df_feat.index.day - 1) / 30.0 - 0.5
    df_feat['month'] = (df_feat.index.month - 1) / 11.0 - 0.5
    
    # --- Price-based features (5m) ---
    df_feat['returns_5m'] = df_feat['Close'].pct_change()
    df_feat['log_returns_5m'] = np.log(df_feat['Close'] / df_feat['Close'].shift(1))
    
    # OHLC relationships
    df_feat['hl_ratio'] = (df_feat['High'] - df_feat['Low']) / df_feat['Close']  # Normalized range
    df_feat['oc_ratio'] = (df_feat['Close'] - df_feat['Open']) / df_feat['Close']  # Body ratio
    df_feat['upper_shadow'] = (df_feat['High'] - df_feat[['Open', 'Close']].max(axis=1)) / df_feat['Close']
    df_feat['lower_shadow'] = (df_feat[['Open', 'Close']].min(axis=1) - df_feat['Low']) / df_feat['Close']
    
    # --- Moving Averages (5m) ---
    for window in [5, 10, 20, 50]:
        df_feat[f'sma_{window}'] = df_feat['Close'].rolling(window=window).mean()
        df_feat[f'sma_{window}_ratio'] = df_feat['Close'] / df_feat[f'sma_{window}'] - 1
    
    # --- Momentum Indicators ---
    # RSI (14 period)
    delta = df_feat['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df_feat['rsi_14'] = 100 - (100 / (1 + rs))
    df_feat['rsi_14_norm'] = df_feat['rsi_14'] / 100.0 - 0.5  # Normalize to [-0.5, 0.5]
    
    # MACD
    exp1 = df_feat['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df_feat['Close'].ewm(span=26, adjust=False).mean()
    df_feat['macd'] = exp1 - exp2
    df_feat['macd_signal'] = df_feat['macd'].ewm(span=9, adjust=False).mean()
    df_feat['macd_hist'] = df_feat['macd'] - df_feat['macd_signal']
    df_feat['macd_norm'] = df_feat['macd'] / df_feat['Close']  # Normalize
    
    # --- Volatility Indicators ---
    df_feat['volatility_20'] = df_feat['log_returns_5m'].rolling(window=20).std()
    df_feat['volatility_50'] = df_feat['log_returns_5m'].rolling(window=50).std()
    
    # ATR (Average True Range)
    high_low = df_feat['High'] - df_feat['Low']
    high_close_prev = (df_feat['High'] - df_feat['Close'].shift(1)).abs()
    low_close_prev = (df_feat['Low'] - df_feat['Close'].shift(1)).abs()
    tr = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
    df_feat['atr_14'] = tr.rolling(window=14).mean()
    df_feat['atr_14_norm'] = df_feat['atr_14'] / df_feat['Close']  # Normalize
    
    # --- Volume Features ---
    df_feat['volume_sma_20'] = df_feat['Volume'].rolling(window=20).mean()
    df_feat['volume_ratio'] = df_feat['Volume'] / df_feat['volume_sma_20']
    
    # --- Multi-Timeframe Features ---
    # Relative position within higher timeframe ranges
    df_feat['price_vs_1h_range'] = (df_feat['Close'] - df_feat['Low_1h']) / (df_feat['High_1h'] - df_feat['Low_1h'] + 1e-8) - 0.5
    df_feat['price_vs_1d_range'] = (df_feat['Close'] - df_feat['Low_1d']) / (df_feat['High_1d'] - df_feat['Low_1d'] + 1e-8) - 0.5
    
    # Higher timeframe returns
    df_feat['return_1h'] = df_feat['Close_1h'].pct_change()
    df_feat['return_1d'] = df_feat['Close_1d'].pct_change()
    
    # Volume ratios across timeframes
    df_feat['volume_5m_vs_1h'] = df_feat['Volume'] / (df_feat['Volume_1h'] / 12 + 1e-8) - 1
    df_feat['volume_5m_vs_1d'] = df_feat['Volume'] / (df_feat['Volume_1d'] / 288 + 1e-8) - 1
    
    # Range expansion/contraction
    df_feat['range_ratio_5m_vs_1h'] = df_feat['Range_5m'] / (df_feat['Range_1h'] / 12 + 1e-8) - 1
    df_feat['range_ratio_5m_vs_1d'] = df_feat['Range_5m'] / (df_feat['Range_1d'] / 288 + 1e-8) - 1
    
    return df_feat

# --- Feature and Target Column Definitions ---
FEATURE_COLUMNS = [
    'hour', 'day_of_week', 'day_of_month', 'month',
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Range_5m', 'Range_1h', 'Range_1d',
    'returns_5m', 'log_returns_5m',
    'hl_ratio', 'oc_ratio', 'upper_shadow', 'lower_shadow',
    'sma_5_ratio', 'sma_10_ratio', 'sma_20_ratio', 'sma_50_ratio',
    'rsi_14_norm', 'macd_norm', 'macd_hist',
    'volatility_20', 'volatility_50', 'atr_14_norm',
    'volume_ratio',
    'price_vs_1h_range', 'price_vs_1d_range',
    'return_1h', 'return_1d',
    'volume_5m_vs_1h', 'volume_5m_vs_1d',
    'range_ratio_5m_vs_1h', 'range_ratio_5m_vs_1d'
]
TARGET_COLUMNS = ['Open', 'High', 'Low', 'Close']
HISTORY_LENGTH = 295  # 288 (context) + 7 (max lag)
print(f'Features: {len(FEATURE_COLUMNS)}, Targets: {len(TARGET_COLUMNS)} (OHLC)')
print(f'HISTORY_LENGTH: {HISTORY_LENGTH}')

print("Feature engineering function defined.")

# Step 5: Data Scaling

## Description:
Scales the 'Close' price using StandardScaler. The scaler is fitted only on the training data to prevent lookahead bias.

In [ ]:
# --- Step 5: Scale the Data (OHLCV + Features) ---
from sklearn.preprocessing import StandardScaler

# === OHLC Scaler (Close'a fit edilir, OHLC'nin 4 kanalina uygulanir) ===
ohlc_cols = ['Open', 'High', 'Low', 'Close']
scaler_close = StandardScaler()

# Train: fit + transform
train_df[ohlc_cols] = scaler_close.fit_transform(train_df[ohlc_cols])

# Val/Test: sadece transform
val_df[ohlc_cols] = scaler_close.transform(val_df[ohlc_cols])
test_df[ohlc_cols] = scaler_close.transform(test_df[ohlc_cols])

# Print (fit'ten SONRA!)
print(f'Close scaler - mean: {scaler_close.mean_[0]:.6f}, std: {scaler_close.scale_[0]:.6f}')

# === Feature Scaler (Volume, Range gibi ek ozellikler icin) ===
feature_scaler_cols = ['Volume', 'Range_5m', 'Range_1h', 'Range_1d']
available_feature_cols = [c for c in feature_scaler_cols if c in train_df.columns]
if available_feature_cols:
    scaler_features = StandardScaler()
    train_df[available_feature_cols] = scaler_features.fit_transform(train_df[available_feature_cols])
    val_df[available_feature_cols] = scaler_features.transform(val_df[available_feature_cols])
    test_df[available_feature_cols] = scaler_features.transform(test_df[available_feature_cols])
    print(f'Feature scaler - features: {available_feature_cols}')


# Step 6: PyTorch Dataset and DataLoader Creation

## Description:
Defines a custom Dataset class to create input/output windows for the model. This class provides the necessary history_length for the model's lag calculations. DataLoaders are then created to manage batching

In [ ]:
# --- Step 6: Dataset ---
class TimeSeriesDataset(Dataset):
    def __init__(self, df, history_length, prediction_length, feature_columns, target_columns):
        self.history_length = history_length
        self.prediction_length = prediction_length
        self.feature_columns = feature_columns
        self.target_columns = target_columns
        self.close_idx = df.columns.get_loc('Close')
        self.target_indices = [df.columns.get_loc(c) for c in target_columns]
        data = torch.from_numpy(df.values).float()
        self.data = torch.nan_to_num(data, nan=0.0)
    def __len__(self):
        return len(self.data) - self.history_length - self.prediction_length + 1
    def __getitem__(self, idx):
        he = idx + self.history_length
        pe = he + self.prediction_length
        return {
            'past_values': self.data[idx:he, self.close_idx],         # [hist_len]
            'past_time_features': self.data[idx:he, :4],              # [hist_len, 4]
            'future_values': self.data[he:pe, self.close_idx],       # [pred_len]
            'future_ohlc': self.data[he:pe][:, self.target_indices], # [pred_len, 4]
            'future_time_features': self.data[he:pe, :4],             # [pred_len, 4]
        }

print('Feature engineering...')
train_features = create_technical_features(train_df).dropna()
val_features = create_technical_features(val_df).dropna()
test_features = create_technical_features(test_df).dropna()
print(f'Train: {len(train_features)}, Val: {len(val_features)}, Test: {len(test_features)}')
train_dataset = TimeSeriesDataset(train_features, HISTORY_LENGTH, PREDICTION_LENGTH, FEATURE_COLUMNS, TARGET_COLUMNS)
val_dataset = TimeSeriesDataset(val_features, HISTORY_LENGTH, PREDICTION_LENGTH, FEATURE_COLUMNS, TARGET_COLUMNS)
test_dataset = TimeSeriesDataset(test_features, HISTORY_LENGTH, PREDICTION_LENGTH, FEATURE_COLUMNS, TARGET_COLUMNS)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print('Datasets ready.')
print(f'Features: {len(FEATURE_COLUMNS)}, Targets: {len(TARGET_COLUMNS)} (OHLC)')
print(f'Train batches: {len(train_dataloader)}')


# Step 7: Model Definition

## Description:
Instantiates the TimeSeriesTransformerForPrediction model using a config object populated with our hyperparameters from Step 2.

In [ ]:
# --- Step 7: Multi-Output Transformer ---
from transformers import TimeSeriesTransformerForPrediction, TimeSeriesTransformerConfig
import torch.nn as nn, torch, copy

class MultiOutputTransformer(nn.Module):
    def __init__(self, base_config, num_outputs=4):
        super().__init__()
        self.base_model = TimeSeriesTransformerForPrediction(base_config)
        hidden_dim = base_config.d_model
        self.ohlc_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(base_config.dropout), nn.Linear(hidden_dim, num_outputs)
        )
    def forward(self, past_values, past_time_features, past_observed_mask=None,
                future_values=None, future_time_features=None, future_observed_mask=None,
                future_ohlc=None):
        outputs = self.base_model(
            past_values=past_values, past_time_features=past_time_features,
            past_observed_mask=past_observed_mask, future_values=future_values,
            future_time_features=future_time_features, future_observed_mask=future_observed_mask,
        )
        # decoder_hidden_states[-1]: [batch, pred_len, d_model] (requires output_hidden_states=True)
        dec_out = outputs.decoder_hidden_states[-1]
        ohlc_pred = self.ohlc_head(dec_out)  # [batch, pred_len, 4]
        return OHLCTransformerOutput(ohlc_pred, outputs)
    def num_params(self):
        return sum(p.numel() for p in self.parameters())

class OHLCTransformerOutput:
    def __init__(self, ohlc_pred, base_outputs):
        self.ohlc = ohlc_pred
        self.base = base_outputs
        self.loss = base_outputs.loss
    @property
    def prediction_outputs(self):
        return self.base.prediction_outputs

config = TimeSeriesTransformerConfig(
    prediction_length=PREDICTION_LENGTH, context_length=CONTEXT_LENGTH,
    lags_sequence=LAGS_SEQUENCE, num_time_features=4,
    num_static_categorical_features=0, distribution_output='student_t', loss='nll',
    encoder_layers=ENCODER_LAYERS, decoder_layers=DECODER_LAYERS,
    d_model=D_MODEL, encoder_attention_heads=ENCODER_ATTENTION_HEADS,
    decoder_attention_heads=DECODER_ATTENTION_HEADS,
    encoder_ffn_dim=ENCODER_FFN_DIM, decoder_ffn_dim=DECODER_FFN_DIM,
    dropout=DROPOUT, output_hidden_states=True,  # KRITIK: decoder hidden'larina erismek icin
)
model = MultiOutputTransformer(config, num_outputs=4)
model.to(device)
print(f'Model created. Outputs: 4 (OHLC). Params: {model.num_params():,}')


# Step 8: Training Loop with Validation

## Description:
The complete training and validation loop. It monitors validation loss and implements early stopping to prevent overfitting, saving the best model state.

In [ ]:
# --- Step 8: Training ---
SPREAD_POINTS = 0.0001
COMMISSION_RATE = 0.00002
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
ohlc_loss_fn = nn.MSELoss()
best_val_loss = float('inf')
patience_counter = 0
for epoch in range(EPOCHS):
    model.train()
    tr_sum = 0.0; ohlc_sum = 0.0; n = 0
    for batch in train_dataloader:
        optimizer.zero_grad()
        out = model(
            past_values=batch['past_values'].to(device),
            past_time_features=batch['past_time_features'].to(device),
            past_observed_mask=torch.ones(batch['past_values'].shape).to(device),
            future_values=batch['future_values'].to(device),
            future_time_features=batch['future_time_features'].to(device),
            future_observed_mask=torch.ones(batch['future_values'].shape).to(device),
            future_ohlc=batch['future_ohlc'].to(device),
        )
        ohlc_loss = ohlc_loss_fn(out.ohlc, batch['future_ohlc'].to(device))
        (out.loss + ohlc_loss).backward()
        optimizer.step()
        tr_sum += out.loss.item() + ohlc_loss.item()
        ohlc_sum += ohlc_loss.item()
        n += 1
    avg_train = tr_sum / n
    avg_ohlc = ohlc_sum / n
    model.eval()
    val_sum = 0.0; vn = 0
    with torch.no_grad():
        for batch in val_dataloader:
            out = model(
                past_values=batch['past_values'].to(device),
                past_time_features=batch['past_time_features'].to(device),
                past_observed_mask=torch.ones(batch['past_values'].shape).to(device),
                future_values=batch['future_values'].to(device),
                future_time_features=batch['future_time_features'].to(device),
                future_observed_mask=torch.ones(batch['future_values'].shape).to(device),
                future_ohlc=batch['future_ohlc'].to(device),
            )
            val_sum += ohlc_loss_fn(out.ohlc, batch['future_ohlc'].to(device)).item()
            vn += 1
    avg_val = val_sum / vn
    print(f'Epoch {epoch+1}/{EPOCHS} - Train: {avg_train:.6f} (OHLC: {avg_ohlc:.6f}) | Val OHLC: {avg_val:.6f}')
    if avg_val < best_val_loss:
        best_val_loss = avg_val; patience_counter = 0
        torch.save(model.state_dict(), 'best_model_multi_output.pth')
        print(f'  -> Best model! Loss: {best_val_loss:.6f}')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}'); break
print('Training done!')

# Step 9: Final Evaluation on All Datasets

## Description:
This comprehensive evaluation step generates predictions for the train, validation, and test sets using the best model saved from the training loop. It then calculates and displays the final RMSE and MAE metrics for all three datasets


In [ ]:
# --- Step 9: Evaluation ---
def get_predictions(loader):
    model.eval()
    all_p, all_a = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                past_values=batch['past_values'].to(device),
                past_time_features=batch['past_time_features'].to(device),
                past_observed_mask=torch.ones(batch['past_values'].shape).to(device),
                future_values=batch['future_values'].to(device),
                future_time_features=batch['future_time_features'].to(device),
                future_observed_mask=torch.ones(batch['future_values'].shape).to(device),
                future_ohlc=batch['future_ohlc'].to(device),
            )
            all_p.append(out.ohlc.cpu().numpy())
            all_a.append(batch['future_ohlc'].cpu().numpy())
    return np.concatenate(all_p, axis=0).reshape(-1, 4), np.concatenate(all_a, axis=0).reshape(-1, 4)
def scale_back(p, a):
    p, a = p.copy(), a.copy()
    for i in range(4):
        p[:, i:i+1] = scaler_close.inverse_transform(p[:, i:i+1])
        a[:, i:i+1] = scaler_close.inverse_transform(a[:, i:i+1])
    return p, a
print('Getting predictions...')
ps, ac = get_predictions(test_dataloader)
p, a = scale_back(ps, ac)
cp, ca = p[:, 3], a[:, 3]
print(f'=== Test (Close) ===')
print(f'RMSE: {np.sqrt(np.mean((cp-ca)**2)):.6f}')
print(f'MAE: {np.mean(np.abs(cp-ca)):.6f}')
print(f'MAPE: {np.mean(np.abs((ca-cp)/(ca+1e-8)))*100:.2f}%')
print(f'Direction Acc: {np.mean(np.sign(np.diff(cp))==np.sign(np.diff(ca)))*100:.1f}%')
print(f'\nOHLC RMSE:')
for i,n in enumerate(['Open','High','Low','Close']):
    print(f'  {n}: {np.sqrt(np.mean((p[:,i]-a[:,i])**2)):.6f}')
print('Done.')


# Step 10: Visualization of Results

## Description:
This cell generates the four key plots that visually compare the model's predictions against the actual values for each dataset, matching the format of your LSTM project.

In [ ]:
# --- Step 10: Visualize Predictions (Corrected for Non-Continuous Time Series) ---
# To avoid plotting gaps (like weekends), we plot against a simple integer sequence
# and then format the x-axis ticks with the corresponding dates. This matches the
# "Gaps Collapsed" approach from the LSTM project.

import matplotlib.dates as mdates

def plot_subset_collapsed(dates, y_true, y_pred, title, subset_size=1000,
                          true_color='royalblue', pred_color='skyblue'):
    """
    Helper function to plot the last N points of a dataset against an integer index,
    collapsing time gaps and adding date labels.
    """
    plt.figure(figsize=(20, 7))

    # Ensure we don't plot more data than we have
    plot_size = min(subset_size, len(dates))

    # Use an integer sequence for the x-axis
    x_axis_index = np.arange(plot_size)

    plt.plot(x_axis_index, y_true[-plot_size:], color=true_color,
             label=f'Actual {title.split(" ")[0]} Price', marker='.', markersize=2, alpha=0.7)
    plt.plot(x_axis_index, y_pred[-plot_size:], color=pred_color,
             label=f'Predicted {title.split(" ")[0]} Price', linestyle='--')

    plt.title(f'{title}: Actual vs. Predicted (Last {plot_size} Points) - Gaps Collapsed', fontsize=16)
    plt.xlabel('Trading Sequence Point (Time Gaps Collapsed)', fontsize=12)
    plt.ylabel('EURUSD Price', fontsize=12)
    plt.legend()
    plt.grid(True)

    # Format the x-axis ticks to show dates
    # We select a few points from our index and label them with the corresponding date
    num_ticks = 7
    tick_indices = np.linspace(0, plot_size - 1, num_ticks, dtype=int)
    tick_labels = [dates[-plot_size:][i].strftime('%Y-%m-%d\n%H:%M') for i in tick_indices]

    plt.xticks(ticks=tick_indices, labels=tick_labels, rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

# --- Plot subsets for clarity ---
plot_subset_collapsed(train_dates, y_true_train, y_pred_train, title='Training Set', true_color='royalblue', pred_color='skyblue')
plot_subset_collapsed(val_dates, y_true_val, y_pred_val, title='Validation Set', true_color='forestgreen', pred_color='lightgreen')
plot_subset_collapsed(test_dates, y_true_test, y_pred_test, title='Test Set', true_color='red', pred_color='darkorange')


# --- Plot 4: Test Set - Scaled (This plot does not need dates, so it's fine as is) ---
# We need to get the scaled true and predicted values for the test set
y_pred_test_scaled = target_scaler.transform(y_pred_test.reshape(-1, 1))
y_true_test_scaled = target_scaler.transform(y_true_test.reshape(-1, 1))

plt.figure(figsize=(20, 7))
plt.plot(y_true_test_scaled, color='blue', label='Actual Test Price (Scaled)')
plt.plot(y_pred_test_scaled, color='lime', label='Predicted Test Price (Scaled)', linestyle='--')
plt.title('Test Set: Actual vs. Predicted EURUSD Price (Scaled)', fontsize=16)
plt.xlabel(f'Time Step (Windowed Test Set)')
plt.ylabel('Scaled Price')
plt.legend()
plt.grid(True)
plt.show()

# Step 11: Save Final Artifacts for Deployment

In [ ]:
# --- Step 11: Save ---
import os
output_dir = 'Models'
torch.save(model.state_dict(), 'Models/model_weights.pth')
joblib.dump(scaler_close, 'Models/scaler_close.pkl')
json.dump(dict(context_length=CONTEXT_LENGTH, prediction_length=PREDICTION_LENGTH,
    lags_sequence=LAGS_SEQUENCE, d_model=D_MODEL, target_columns=TARGET_COLUMNS,
    scaler_mean=scaler_close.mean_.tolist(), scaler_std=scaler_close.scale_.tolist()),
    open('Models/config.json', 'w'), indent=2)
print('All artifacts saved to Models/')